# BERT 임베딩으로 문장 유사도 계산하기

**Tutorial:** BERT: Pre-training of Deep Bidirectional Transformers  
**Section:** 실습 코드 3 — BERT 임베딩으로 문장 유사도 계산

---

## 이 노트북에서 배울 것

| # | 개념 | 한 줄 요약 |
|---|------|------------|
| 1 | 토크나이저 | 텍스트 문자열을 BERT가 읽을 수 있는 숫자로 바꾸는 과정 |
| 2 | `[CLS]` 토큰 | 문장 전체 의미를 한 벡터에 압축하는 특수 토큰 |
| 3 | 임베딩 | 단어·문장을 의미가 담긴 숫자 배열(벡터)로 표현하는 것 |
| 4 | 코사인 유사도 | 두 벡터가 얼마나 같은 방향을 가리키는지 수치로 측정 |

---

## 전체 흐름 한눈에 보기

```
텍스트 문장
    │
    ▼  (1) 토크나이저
[CLS] token1 token2 ... [SEP]   ← 특수 토큰이 자동으로 붙음
    │
    ▼  (2) 토큰 → 숫자 ID 변환
[ 101,  1037,  3899,  2003, ... , 102 ]
    │
    ▼  (3) BERT 모델 (12층 Transformer)
각 토큰마다 768차원 벡터 출력
    │
    ▼  (4) 위치 0번([CLS]) 벡터만 추출
문장 임베딩 (shape: 768,)
    │
    ▼  (5) L2 정규화 (크기를 1로)
단위 벡터
    │
    ▼  (6) 두 문장 벡터의 내적 = 코사인 유사도
유사도 점수 (−1 ~ 1)
```

---

## 사전 지식 체크
- ✅ 파이썬 기초 (함수, for문, 리스트)
- ✅ '벡터 = 숫자들의 배열'이라는 직관
- ⬜ BERT 논문 내용 → 몰라도 됩니다. 이 노트북이 친절하게 설명합니다.

---
## 💡 잠깐! BERT를 한 줄로 이해하고 시작하기

BERT는 **수십억 개의 문장을 읽으며** 단어/문장의 의미를 숫자로 표현하는 법을 스스로 학습한 모델입니다.

학습이 끝난 BERT는 아래처럼 동작합니다:

```
"강아지가 공원에서 뛰어놀고 있다."
    → BERT →  [ 0.23, -0.15,  0.88, 0.04, ... ]  ← 768개 숫자

"개가 야외에서 놀고 있다."
    → BERT →  [ 0.25, -0.14,  0.86, 0.05, ... ]  ← 비슷한 의미 = 비슷한 숫자!

"오늘 주식시장이 폭락했다."
    → BERT →  [-0.10,  0.72, -0.03, 0.90, ... ]  ← 다른 의미 = 다른 숫자
```

이 '비슷한 의미 → 비슷한 숫자'라는 성질을 이용해 **문장 유사도**를 측정합니다.

In [ ]:
# ============================================================
# Step 0: 라이브러리 설치 (처음 실행하는 경우)
# ============================================================
# Google Colab 또는 새 환경에서 실행 중이라면 아래 두 줄의 '#'을 지우고 실행하세요.
# 이미 설치되어 있다면 이 셀은 건너뛰어도 됩니다.

# !pip install transformers torch  # BERT 모델을 사용하기 위한 라이브러리
# !pip install matplotlib numpy    # 시각화 라이브러리

In [ ]:
# ============================================================
# Step 1: 라이브러리 불러오기
# ============================================================
# 각 라이브러리가 어떤 역할을 하는지 확인하면서 읽으세요.

from transformers import BertTokenizer, BertModel
# transformers: Hugging Face에서 만든 라이브러리.
#   BertTokenizer : 텍스트 → 토큰 ID 변환 담당
#   BertModel     : 실제 BERT 신경망 (토큰 ID → 벡터 변환 담당)

import torch
# PyTorch: 텐서(tensor, 다차원 배열) 연산과 딥러닝을 위한 라이브러리.
# '텐서' = 넘파이 배열과 비슷하지만 GPU 계산, 자동 미분 등을 지원합니다.

import torch.nn.functional as F
# 자주 쓰는 함수들의 모음 (여기서는 normalize() 함수 사용)

import numpy as np
# 행렬 계산 및 데이터 처리용

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
# 유사도 행렬(히트맵) 시각화용

print("✅ 라이브러리 불러오기 완료!")
print(f"   PyTorch 버전: {torch.__version__}")

---
## Part 1. BERT 모델 준비하기

In [ ]:
# ============================================================
# Step 2: BERT 토크나이저 & 모델 로드
# ============================================================
#
# 'bert-base-uncased' 란?
#   - bert-base : 12개 Transformer 레이어, 768 은닉 차원, 1.1억 파라미터
#                 (bert-large보다 작지만 실습용으로 충분)
#   - uncased   : 대소문자를 구별하지 않음 → 입력을 모두 소문자로 변환
#                 ('Dog' 와 'dog' 을 동일하게 처리)
#
# from_pretrained() : Hugging Face Hub에서 학습된 가중치를 다운로드합니다.
#   ⚠️ 처음 실행 시 약 440MB를 다운로드하므로 시간이 걸릴 수 있습니다.
#      이후에는 캐시에서 즉시 불러옵니다.

print("BERT 모델 로드 중... (처음 실행 시 다운로드 필요)")
print()

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# 역할: 텍스트 문자열을 숫자 ID의 배열로 변환
# 내부적으로 WordPiece 사전(어휘 30,522개)을 가지고 있습니다.

model = BertModel.from_pretrained('bert-base-uncased')
# 역할: 숫자 ID 배열을 받아 각 토큰의 의미 벡터를 출력

model.eval()
# eval() 모드로 전환하는 이유:
#   BERT 내부에는 학습 중에만 사용하는 레이어(예: Dropout)가 있습니다.
#   → 학습(train) 모드: Dropout이 무작위로 뉴런을 끔 (과적합 방지)
#   → 평가(eval)  모드: Dropout을 끄고 모든 뉴런을 사용 (일관된 결과)
#   우리는 추론만 할 것이므로 반드시 eval() 호출 필요!

print("✅ 모델 로드 완료!")
total_params = sum(p.numel() for p in model.parameters())
print(f"   BERT-base 총 파라미터 수: {total_params:,}개 (약 {total_params/1e6:.0f}M)")
# 참고: BERT-base ≈ 110M 파라미터

---
## Part 2. 토크나이저 이해하기 (Step-by-Step)

BERT는 텍스트를 직접 읽지 않습니다.  
**텍스트 → 토큰 → 숫자 ID → 텐서** 순서로 변환한 후 처리합니다.  
이 과정을 단계별로 살펴봅시다.

In [ ]:
# ============================================================
# Step 3-A: 1단계 — 텍스트를 토큰으로 쪼개기 (Tokenization)
# ============================================================

example_text = "A dog is running in the park."

tokens = tokenizer.tokenize(example_text)
print("[원문]", example_text)
print("[토큰]", tokens)
print(f"  → 총 {len(tokens)}개의 토큰으로 분리됨")
print()

# BERT는 WordPiece 방식을 사용합니다.
# → 모르는 단어는 더 작은 단위로 쪼갭니다.
# → 앞 토큰과 이어지는 조각에는 '##' 접두사가 붙습니다.

# WordPiece 분리 예시 확인
print("[WordPiece 분리 예시 — 드문 단어는 조각으로 나뉩니다]")
test_words = ["running", "unbelievable", "transformer", "embedding"]
for word in test_words:
    pieces = tokenizer.tokenize(word)
    print(f"  '{word}' → {pieces}")

In [ ]:
# ============================================================
# Step 3-B: 2단계 — 특수 토큰 추가 & 숫자 ID로 변환
# ============================================================
#
# tokenizer()를 호출하면 세 가지 작업이 한 번에 수행됩니다:
#   ① 토큰화
#   ② [CLS]와 [SEP] 특수 토큰 추가
#   ③ 각 토큰을 사전의 숫자 ID로 변환

inputs = tokenizer(
    example_text,
    return_tensors="pt"  # "pt" = PyTorch 텐서 형식으로 반환 ("tf" 이면 TensorFlow)
)

# 변환된 ID를 다시 토큰 이름으로 되돌려 확인
token_ids  = inputs['input_ids'][0].tolist()
token_strs = tokenizer.convert_ids_to_tokens(token_ids)

print("[특수 토큰 포함, 최종 입력 구조]")
print(f"{'위치':>4}  {'토큰':>12}  {'ID':>6}  {'의미':<30}")
print("-" * 60)
for pos, (tok, tid) in enumerate(zip(token_strs, token_ids)):
    if tok == '[CLS]':
        note = "← 문장 시작 / 문장 전체 의미를 담는 토큰!"
    elif tok == '[SEP]':
        note = "← 문장 끝을 알리는 구분 토큰"
    else:
        note = ""
    print(f"  {pos:>2}  {tok:>12}  {tid:>6}  {note}")

print()
print("inputer 출력 딕셔너리의 key별 의미:")
for key, val in inputs.items():
    print(f"  '{key}': shape={tuple(val.shape)}")
    if key == 'input_ids':
        print(f"    → 각 토큰의 어휘 사전 ID (0~30521 범위)")
    elif key == 'token_type_ids':
        print(f"    → 0=첫 번째 문장, 1=두 번째 문장 (단일 문장이면 모두 0)")
    elif key == 'attention_mask':
        print(f"    → 1=실제 토큰, 0=패딩(빈칸) — 현재는 모두 1")

---
## Part 3. `[CLS]` 토큰과 임베딩 추출

### `[CLS]` 토큰이 왜 '문장 대표 벡터'가 될 수 있나요?

BERT의 Transformer 레이어는 **Self-Attention**을 사용합니다.  
Self-Attention 덕분에 `[CLS]` 토큰은 문장 내 **모든 다른 토큰을 참조**하면서 자신의 벡터를 업데이트합니다.

```
레이어 1 →  [CLS]' = Attention(CLS, dog, is, running, park, ...)
레이어 2 →  [CLS]'' = Attention([CLS]', dog', is', ...)
   ...
레이어 12 → [CLS](12) ← 이 시점의 벡터가 전체 문장을 압축한 표현!
```

BERT는 사전 학습 과정에서 `[CLS]` 벡터로 문장 수준의 과제(NSP: Next Sentence Prediction)를 수행하도록 훈련되었기 때문에,  
자연스럽게 문장 전체의 의미 정보를 흡수하게 됩니다.

In [ ]:
# ============================================================
# Step 4: BERT 모델을 통과시켜 임베딩 추출 과정 직접 관찰
# ============================================================

inputs = tokenizer(
    example_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# torch.no_grad() 블록 안에서는 그래디언트를 계산하지 않습니다.
#   - 그래디언트(gradient): 역전파 학습 때 가중치를 업데이트하기 위한 값
#   - 우리는 학습이 아닌 추론(inference)만 하므로 불필요한 계산을 끄는 것
#   - 메모리와 속도 모두 절약됩니다.
with torch.no_grad():
    outputs = model(**inputs)
    # **inputs 는 딕셔너리를 키워드 인자로 풀어서 전달하는 문법:
    # model(input_ids=..., token_type_ids=..., attention_mask=...) 와 동일

# ── 출력 텐서의 shape 이해하기 ──
# outputs.last_hidden_state : 마지막 (12번째) Transformer 레이어의 출력
# shape = (batch_size, sequence_length, hidden_size)
#          배치 크기  ,  시퀀스 길이    ,  은닉 차원(768)
last_hidden = outputs.last_hidden_state
print("[BERT 최종 출력 텐서 shape]")
print(f"  last_hidden_state.shape = {last_hidden.shape}")
batch_size, seq_len, hidden_size = last_hidden.shape
print(f"  ├─ batch_size  = {batch_size}  (한 번에 처리한 문장 수)")
print(f"  ├─ seq_len     = {seq_len}  (토큰 수: [CLS] + {seq_len-2} 단어 + [SEP])")
print(f"  └─ hidden_size = {hidden_size} (각 토큰당 768차원 벡터)")
print()

# [CLS] 토큰 임베딩 추출:
#   last_hidden_state[:, 0, :]
#   ─────────────────────────
#   : → 배치 전체 (배치 크기 1이므로 1개)
#   0 → 시퀀스 위치 0번 = [CLS] 토큰 !
#   : → 768개 값 전부
cls_embedding = last_hidden[:, 0, :]
print(f"[CLS] 임베딩 shape: {cls_embedding.shape}")
print(f"  → 이 (1, 768) 벡터 하나가 문장 전체를 대표합니다.")
print()
print(f"처음 10개 원소 미리보기:")
print(f"  {cls_embedding[0, :10].numpy().round(4)}")

---
## Part 4. 코사인 유사도 (Cosine Similarity) 이해하기

두 벡터 A, B 사이의 코사인 유사도는 다음과 같이 정의됩니다:

$$
\text{cosine\_sim}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}
$$

- **분자 $A \cdot B$**: 두 벡터의 내적 (같은 위치 원소끼리 곱해서 합산)
- **분모 $\|A\| \times \|B\|$**: 두 벡터 크기의 곱
- 결과는 **−1 ~ 1** 사이의 값

**정규화(normalize)를 하면 계산이 단순해집니다:**  
벡터 크기를 미리 1로 만들면 → 분모가 항상 1 × 1 = 1  
따라서 `cos_sim = A · B` (그냥 내적!)

| 유사도 값 | 의미 |
|-----------|------|
| **1.0** | 완전히 같은 방향 (의미가 동일) |
| **0.7 ~ 1.0** | 비슷한 의미 |
| **0.5 ~ 0.7** | 약간 관련 있음 |
| **0.5 미만** | 관련 없음 |
| **−1.0** | 완전히 반대 방향 (의미가 반대) |

In [ ]:
# ============================================================
# Step 5: 코사인 유사도를 2D 벡터로 먼저 직관적으로 이해하기
# ============================================================
# 실제 BERT는 768차원이지만, 개념은 2D에서도 완전히 동일합니다.

def cosine_similarity(a, b):
    """
    두 벡터의 코사인 유사도를 계산합니다.
    정규화 후 내적(dot product)으로 구합니다.
    """
    # unsqueeze(0): (768,) → (1, 768) 로 차원 추가 (normalize 함수 요구사항)
    # squeeze(): 다시 (768,) 로 줄이기
    a_norm = F.normalize(a.unsqueeze(0), dim=-1).squeeze()
    b_norm = F.normalize(b.unsqueeze(0), dim=-1).squeeze()
    return torch.dot(a_norm, b_norm).item()

# 2D 벡터 예시 (x, y 좌표계)
cases = [
    (torch.tensor([1.0, 0.0]),  torch.tensor([1.0, 0.0]),  "완전히 같은 방향 (동일 벡터)"),
    (torch.tensor([1.0, 0.0]),  torch.tensor([2.0, 0.5]),  "거의 같은 방향 (크기만 다름)"),
    (torch.tensor([1.0, 0.0]),  torch.tensor([0.7, 0.7]),  "45도 차이"),
    (torch.tensor([1.0, 0.0]),  torch.tensor([0.0, 1.0]),  "수직 (완전 무관)"),
    (torch.tensor([1.0, 0.0]),  torch.tensor([-1.0, 0.0]), "정반대 방향"),
]

print("=" * 65)
print("2D 벡터로 보는 코사인 유사도")
print("=" * 65)
for a, b, desc in cases:
    sim = cosine_similarity(a, b)
    bar_len = int((sim + 1) / 2 * 20)   # -1~1 범위를 0~20 막대로 표현
    bar = '█' * bar_len + '░' * (20 - bar_len)
    print(f"  {sim:+.3f} |{bar}| {desc}")

print()
print("💡 BERT의 문장 임베딩도 같은 방식으로 비교합니다. 차원만 768개로 늘어날 뿐!")

---
## Part 5. 문장 임베딩 추출 함수 만들기

In [ ]:
# ============================================================
# Step 6: 문장 임베딩 추출 함수 정의
# ============================================================
#
# 앞에서 Step-by-Step으로 살펴본 과정을 함수로 정리합니다.

def get_sentence_embedding(text):
    """
    텍스트 문장을 BERT [CLS] 임베딩 벡터로 변환합니다.

    Args:
        text (str): 임베딩을 추출할 문장

    Returns:
        torch.Tensor: shape (1, 768), L2 정규화된 단위 벡터
                      (크기 = 1 이므로 내적 = 코사인 유사도)
    """

    # ── 1단계: 텍스트 → 입력 텐서 변환 ──────────────────────────
    inputs = tokenizer(
        text,
        return_tensors="pt",  # PyTorch 텐서로 반환
        truncation=True,      # 512 토큰 초과 시 뒷부분을 자름
        max_length=512        # BERT가 처리할 수 있는 최대 길이
    )
    # inputs = {
    #   'input_ids':      (1, seq_len) — 토큰 ID
    #   'token_type_ids': (1, seq_len) — 문장 구분 (모두 0)
    #   'attention_mask': (1, seq_len) — 실제 토큰 여부 (모두 1)
    # }

    # ── 2단계: BERT 모델 통과 ─────────────────────────────────────
    with torch.no_grad():           # 그래디언트 계산 비활성화 (추론 전용)
        outputs = model(**inputs)   # BERT의 12개 레이어를 순차적으로 통과
    # outputs.last_hidden_state.shape = (1, seq_len, 768)

    # ── 3단계: [CLS] 토큰(위치 0) 벡터 추출 ─────────────────────
    cls_embedding = outputs.last_hidden_state[:, 0, :]
    # [:, 0, :] → 배치 전체, 위치 0번([CLS]), 모든 차원
    # shape: (1, 768)

    # ── 4단계: L2 정규화 (벡터 크기를 1로 만들기) ─────────────────
    normalized = F.normalize(cls_embedding, dim=-1)
    # dim=-1 : 마지막 차원(768차원)을 기준으로 정규화
    # 정규화 전: ||cls_embedding|| = 어떤 값
    # 정규화 후: ||normalized||    = 1.0  ← 단위 벡터
    # 이후 두 벡터의 내적(dot product) = 코사인 유사도

    return normalized


# ── 함수 검증 ─────────────────────────────────────────────────────
print("[함수 동작 검증]")
test_emb = get_sentence_embedding("Hello, BERT!")
print(f"  출력 shape : {test_emb.shape}  (batch=1, dim=768)")
print(f"  벡터 크기  : {test_emb.norm().item():.6f}  ← 정규화 후이므로 반드시 1.0이어야 함")
print(f"  처음 5개 값: {test_emb[0, :5].numpy().round(4)}")
print()
print("✅ get_sentence_embedding() 함수 준비 완료")

---
## Part 6. 실제 문장들로 유사도 계산하기

In [ ]:
# ============================================================
# Step 7: 실험용 문장 준비 & 임베딩 추출
# ============================================================
#
# 의도적으로 '비슷한 그룹'과 '다른 그룹'을 섞어 준비했습니다.
# BERT가 의미 유사성을 잘 잡아내는지 확인해 봅시다.
#
#  🐶 그룹 A (강아지/동물) : 문장 0, 1, 4
#  📈 그룹 B (경제/주식)   : 문장 2, 5
#  🎵 그룹 C (음악)        : 문장 3
#  🍕 그룹 D (음식)        : 문장 6, 7

sentences = [
    # 그룹 A — 강아지/동물 (서로 유사해야 함)
    "A dog is running in the park.",              # 0
    "A puppy is playing outside.",                # 1 — 0과 거의 같은 의미
    "My pet is chasing a ball in the garden.",    # 4 — 0, 1과 비슷

    # 그룹 B — 경제/주식 (서로 유사해야 함)
    "The stock market crashed today.",             # 2
    "Investors are worried about the economy.",   # 5 — 2와 비슷

    # 그룹 C — 음악 (다른 그룹과 달라야 함)
    "I love listening to classical music.",        # 3

    # 그룹 D — 음식 (서로 유사, 다른 그룹과 달라야 함)
    "I am very hungry right now.",                # 6
    "Let's go find a nice restaurant for dinner.",# 7
]

# 그룹 레이블 (시각화용)
group_labels = ['🐶A', '🐶A', '🐶A', '📈B', '📈B', '🎵C', '🍕D', '🍕D']

# 각 문장의 임베딩 추출
print("임베딩 추출 중...")
embeddings = []
for i, sentence in enumerate(sentences):
    emb = get_sentence_embedding(sentence)
    embeddings.append(emb)
    print(f"  [{i}] {group_labels[i]}  '{sentence[:50]}'")

print(f"\n✅ 총 {len(embeddings)}개 문장의 임베딩 추출 완료")
print(f"   각 임베딩의 shape: {embeddings[0].shape}")

In [ ]:
# ============================================================
# Step 8: 문장 쌍(pair)별 유사도 계산 및 출력
# ============================================================

def similarity_label(score):
    """유사도 점수에 따라 직관적인 레이블 반환"""
    if score >= 0.90:  return "🔴 매우 유사"
    elif score >= 0.80: return "🟠 유사"
    elif score >= 0.70: return "🟡 약간 유사"
    elif score >= 0.60: return "🟢 약간 관련"
    else:               return "🔵 관련 없음"

print("=" * 68)
print("BERT 문장 유사도 결과 (Cosine Similarity)")
print("=" * 68)

n = len(sentences)
for i in range(n):
    for j in range(i + 1, n):
        # ─ 핵심 계산 ────────────────────────────────────────────
        # .squeeze(): (1, 768) → (768,) 로 불필요한 차원 제거
        # torch.dot(): 두 1D 벡터의 내적 (= 정규화 후이므로 코사인 유사도)
        # .item(): 0차원 PyTorch 텐서를 파이썬 float로 변환
        sim = torch.dot(
            embeddings[i].squeeze(),
            embeddings[j].squeeze()
        ).item()
        # ──────────────────────────────────────────────────────────

        label = similarity_label(sim)
        same_group = "✅ 같은 그룹" if group_labels[i][1] == group_labels[j][1] else ""

        print(f"[{i}↔{j}] 유사도: {sim:.4f}  {label}  {same_group}")
        print(f"   [{i}] {group_labels[i]} '{sentences[i]}'")
        print(f"   [{j}] {group_labels[j]} '{sentences[j]}'")
        print()

print("💡 같은 그룹(✅)일수록 높은 유사도, 다른 그룹일수록 낮은 유사도가 나타나야 합니다.")

In [ ]:
# ============================================================
# Step 9: 유사도 행렬(Similarity Matrix) 히트맵 시각화
# ============================================================
#
# 모든 문장 쌍의 유사도를 N×N 표(행렬)로 만들어 한눈에 봅니다.
# 같은 그룹 문장들끼리 모여있는 밝은 구역이 대각 블록 형태로 나타나야 합니다.

n = len(sentences)

# N×N 유사도 행렬 생성
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        # 자기 자신(i==j)은 항상 1.0 (완전히 동일)
        sim_matrix[i][j] = torch.dot(
            embeddings[i].squeeze(),
            embeddings[j].squeeze()
        ).item()

# 축 레이블 (짧게 줄인 버전)
short_labels = [
    f"[0]{group_labels[0]} dog/park",
    f"[1]{group_labels[1]} puppy/outside",
    f"[2]{group_labels[2]} pet/garden",
    f"[3]{group_labels[3]} stock market",
    f"[4]{group_labels[4]} investors",
    f"[5]{group_labels[5]} music",
    f"[6]{group_labels[6]} hungry",
    f"[7]{group_labels[7]} restaurant",
]

fig, ax = plt.subplots(figsize=(10, 8))

# 히트맵 그리기
im = ax.imshow(
    sim_matrix,
    cmap='RdYlGn',   # 빨강(낮음) → 노랑 → 초록(높음)
    vmin=0.4,        # 색상 최솟값 (BERT 임베딩은 보통 0.4 이상)
    vmax=1.0         # 색상 최댓값
)

# 눈금 & 레이블
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(short_labels, rotation=35, ha='right', fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)

# 각 셀에 숫자 표시
for i in range(n):
    for j in range(n):
        val = sim_matrix[i][j]
        # 배경 밝기에 따라 글씨 색 선택
        text_color = 'black' if val > 0.65 else 'white'
        fontw = 'bold' if i == j else 'normal'
        ax.text(
            j, i, f"{val:.3f}",
            ha='center', va='center',
            fontsize=8.5, color=text_color, fontweight=fontw
        )

# 같은 그룹 영역에 테두리 표시
# 그룹 A: 0-1-2, 그룹 B: 3-4, 그룹 C: 5, 그룹 D: 6-7
group_ranges = [(0, 2), (3, 4), (5, 5), (6, 7)]
for (start, end) in group_ranges:
    size = end - start + 1
    rect = plt.Rectangle(
        (start - 0.5, start - 0.5), size, size,
        fill=False, edgecolor='navy', linewidth=2.5, linestyle='--'
    )
    ax.add_patch(rect)

plt.colorbar(im, ax=ax, label='Cosine Similarity', shrink=0.8)
ax.set_title(
    'BERT CLS 임베딩 문장 유사도 행렬\n'
    '(파란 점선 박스 = 같은 의미 그룹, 밝을수록 유사)',
    fontsize=12, pad=15
)
plt.tight_layout()
plt.show()

print()
print("💡 파란 점선 박스 안(같은 그룹)의 값이 박스 밖(다른 그룹)보다 높은지 확인하세요!")
print("   BERT가 단어 겹침이 없어도 의미 유사성을 잘 포착하고 있습니다.")

---
## Part 7. 흥미로운 실험들

In [ ]:
# ============================================================
# Step 10: 실험 1 — 동의어 vs 반의어
# ============================================================
# BERT가 의미를 얼마나 잘 이해하는지 확인해봅니다.

print("=" * 60)
print("실험 1: 기준 문장과 여러 표현의 유사도 비교")
print("=" * 60)

anchor     = "The cat sat on the mat."
comparisons = [
    ("The cat sat on the mat.",       "완전 동일 문장 → 1.0이어야 함"),
    ("A feline rested on a rug.",      "동의어 사용, 같은 의미 → 높아야 함"),
    ("The kitten is lying on the floor.", "비슷한 상황 → 중간 정도"),
    ("Dogs love to play fetch outside.", "동물이지만 다른 상황 → 낮아야 함"),
    ("The president signed a new law.",  "완전히 다른 주제 → 가장 낮아야 함"),
]

anchor_emb = get_sentence_embedding(anchor)
print(f"기준 문장: '{anchor}'\n")

for comp_text, expected in comparisons:
    comp_emb = get_sentence_embedding(comp_text)
    sim = torch.dot(anchor_emb.squeeze(), comp_emb.squeeze()).item()

    # 막대 그래프 (0.4 ~ 1.0 범위)
    bar_len = max(0, int((sim - 0.4) / 0.6 * 20))
    bar = '█' * bar_len + '░' * (20 - bar_len)

    print(f"  유사도: {sim:.4f}  |{bar}|")
    print(f"  비교   : '{comp_text}'")
    print(f"  예상   : {expected}")
    print()

In [ ]:
# ============================================================
# Step 11: 실험 2 — 길이 차이가 유사도에 미치는 영향
# ============================================================
# 짧은 문장과 긴 문장의 유사도는 어떻게 될까요?

print("=" * 60)
print("실험 2: 문장 길이 차이와 유사도")
print("=" * 60)

anchor2 = "Machine learning is interesting."
length_tests = [
    "ML is cool.",
    "Machine learning is really interesting.",
    "Machine learning is a fascinating field that involves training algorithms on data to recognize patterns and make predictions.",
    "Deep learning, a subset of machine learning, uses neural networks with many layers to process complex data like images and text.",
]

anchor2_emb = get_sentence_embedding(anchor2)
print(f"기준 문장: '{anchor2}'")
print(f"(토큰 수: {len(tokenizer.tokenize(anchor2))}개)\n")

for test_text in length_tests:
    test_emb = get_sentence_embedding(test_text)
    sim = torch.dot(anchor2_emb.squeeze(), test_emb.squeeze()).item()
    tok_count = len(tokenizer.tokenize(test_text))
    print(f"  유사도: {sim:.4f}  (토큰 {tok_count:3d}개)  '{test_text[:65]}'")

In [ ]:
# ============================================================
# Step 12: 실험 3 — 직접 나만의 문장으로 실험해보기
# ============================================================
# 아래 my_sentences 리스트를 자유롭게 수정해보세요!
#
# 추천 실험:
#   A) 같은 뜻, 다른 표현: "I'm starving" vs "I haven't eaten all day"
#   B) 동음이의어:          "I went to the bank" (은행? 강둑?)
#   C) 긍정 vs 부정:        "I love this movie" vs "I hate this movie"
#   D) 나라 언어 혼합:       영어 문장 vs 한국어 문장 (bert-base-uncased는 영어 전용)

my_sentences = [
    "I am absolutely starving right now.",
    "I haven't eaten anything all day and I'm really hungry.",
    "Let's go grab some food somewhere.",
    "The concert was absolutely amazing last night.",
]

# ── 임베딩 추출 ───────────────────────────────────────────────
my_embeddings = [get_sentence_embedding(s) for s in my_sentences]

# ── 결과 출력 ────────────────────────────────────────────────
print("=" * 62)
print("나만의 문장 유사도 실험 결과")
print("=" * 62)
for i, s in enumerate(my_sentences):
    print(f"  [{i}] {s}")
print()

for i in range(len(my_sentences)):
    for j in range(i + 1, len(my_sentences)):
        sim = torch.dot(
            my_embeddings[i].squeeze(),
            my_embeddings[j].squeeze()
        ).item()
        bar_len = max(0, int((sim - 0.4) / 0.6 * 25))
        bar = '█' * bar_len + '░' * (25 - bar_len)
        print(f"[{i}↔{j}] {sim:.4f}  |{bar}|  {similarity_label(sim)}")
print()
print("✏️  my_sentences 안의 문장들을 바꿔가며 BERT의 반응을 관찰해보세요!")

---
## Part 8. 정리 & 다음 단계

### 오늘 배운 전체 흐름 복습

```
텍스트 문장
    │
    ▼  tokenizer(text, return_tensors='pt')
input_ids, token_type_ids, attention_mask   ← 숫자 텐서
    │
    ▼  model(**inputs)  [torch.no_grad() 안에서]
outputs.last_hidden_state   shape: (1, seq_len, 768)
    │
    ▼  [:, 0, :]  ← [CLS] 위치만 추출
cls_embedding   shape: (1, 768)
    │
    ▼  F.normalize(dim=-1)
단위 벡터 (크기=1)
    │
    ▼  torch.dot(emb_A, emb_B)
코사인 유사도   범위: -1 ~ 1
```

### 핵심 키워드 정리

| 키워드 | 의미 |
|--------|------|
| **WordPiece 토크나이저** | 드문 단어를 더 작은 조각으로 분리하는 방식 (`playing` → `play`, `##ing`) |
| **`[CLS]` 토큰** | 문장 앞에 붙는 특수 토큰. BERT 통과 후 문장 전체 의미를 담음 |
| **`[SEP]` 토큰** | 문장 끝을 나타내는 구분 토큰 |
| **임베딩(Embedding)** | 의미를 담은 숫자 벡터 표현 |
| **L2 정규화** | 벡터 크기를 1로 만드는 것 → 이후 내적 = 코사인 유사도 |
| **코사인 유사도** | 두 벡터의 방향 유사성 (-1 ~ 1, 높을수록 의미 유사) |

### 다음으로 공부할 주제

1. **Mean Pooling vs CLS Pooling** — `[CLS]` 대신 모든 토큰의 평균을 쓰면 어떨까?
2. **Sentence-BERT (SBERT)** — 문장 유사도 전용으로 파인튜닝된 BERT
3. **Semantic Search** — 임베딩을 활용한 의미 기반 검색
4. **벡터 데이터베이스** — 수백만 개의 임베딩을 빠르게 검색하는 방법 (FAISS, Pinecone 등)